In [38]:
import pandas as pd
import numpy as np

In [39]:
import os
print(os.listdir("crime_data/crime"))

['01_District_wise_crimes_committed_IPC_2001_2012.csv', '01_District_wise_crimes_committed_IPC_2013.csv', '01_District_wise_crimes_committed_IPC_2014.csv', '02_01_District_wise_crimes_committed_against_SC_2001_2012.csv', '02_01_District_wise_crimes_committed_against_SC_2013.csv', '02_01_District_wise_crimes_committed_against_SC_2014.csv', '02_District_wise_crimes_committed_against_ST_2001_2012.csv', '02_District_wise_crimes_committed_against_ST_2013.csv', '02_District_wise_crimes_committed_against_ST_2014.csv', '03_District_wise_crimes_committed_against_children_2001_2012.csv', '03_District_wise_crimes_committed_against_children_2013.csv', '03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2012.csv', '03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2013.csv', '03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2014.csv', '04_01_Person_arrested_and_their_disposal_by_police_

In [40]:
ipc = pd.read_csv("crime_data/crime/IPC_2001to2014.csv")
women1 = pd.read_csv("crime_data/crime/women_2001to 2013.csv")
women2 = pd.read_csv("crime_data/crime/42_District_wise_crimes_committed_against_women_2014.csv")
children = pd.read_csv("crime_data/crime/Children_2001to2013.csv")
police = pd.read_csv("crime_data/crime/12_Police_strength_actual_and_sanctioned.csv")
arrests = pd.read_csv("crime_data/crime/arrests2012to2014.csv")

In [41]:
def clean(df):
    df.columns = df.columns.str.strip().str.lower()
    return df

ipc = clean(ipc)
women1 = clean(women1)
women2 = clean(women2)
children = clean(children)
police = clean(police)
arrests = clean(arrests)

In [42]:
def fix_state(df):
    df["state"] = df["state"].astype(str).str.strip().str.upper()
    df["state"] = df["state"].str.replace("&", "AND")
    return df

In [43]:
# IPC
ipc = ipc.rename(columns={"state/ut": "state"})

# Women
women1 = women1.rename(columns={"state/ut": "state"})
women2 = women2.rename(columns={"states/uts": "state"})

# Children
children = children.rename(columns={"state/ut": "state"})

# Police
police = police.rename(columns={"area_name": "state"})

# Arrest
arrests = arrests.rename(columns={
    "state/ut": "state",
    "persons arrested during the year": "total_arrests"
})

In [44]:
def fix_state(df):
    df["state"] = df["state"].astype(str).str.strip().str.upper()
    df["state"] = df["state"].str.replace("&", "AND")
    return df

In [45]:
ipc = fix_state(ipc)
women1 = fix_state(women1)
women2 = fix_state(women2)
children = fix_state(children)
police = fix_state(police)
arrests = fix_state(arrests)

In [46]:
ipc["ipc_total"] = ipc.select_dtypes(include="number").sum(axis=1)
ipc = ipc[["state","district","year","ipc_total"]]


In [47]:
women1["women_total"] = women1.select_dtypes(include="number").sum(axis=1)
women1 = women1[["state","district","year","women_total"]]

women = pd.concat([women1, women2], ignore_index=True)


In [48]:
children["children_total"] = children.select_dtypes(include="number").sum(axis=1)
children = children[["state","district","year","children_total"]]


In [49]:
print(police.columns.tolist())

['state', 'year', 'group_name', 'sub_group_name', 'rank_all_ranks_total', 'rank_asi_equivalent', 'rank_aspdyspassttcommandant', 'rank_below_hc_and_above_constables', 'rank_constables', 'rank_dgaddl_dg', 'rank_dig', 'rank_head_constables', 'rank_igsplig', 'rank_inspectors_equivalent', 'rank_si_equivalent', 'rank_sspspaddlspcommandant']


In [50]:
police = police.rename(columns={
    "rank_all_ranks_total": "total_police"
})

In [51]:
police = police[["state","year","total_police"]]
police = police.groupby(["state","year"])["total_police"].sum().reset_index()

In [52]:
print(arrests.columns.tolist())

['state', 'crime head', 'persons in custody or on bail during the stage of investigation at the beginning of the year', 'total_arrests', 'persons released or freed by police or magistrate before trial for want of evidence or any other reason', 'persons in custody or on bail during the stage of investigation at the end of the year', 'persons in whose cases charge sheets were laid during the year', 'persons under trial at the beginning of the year', 'total number of persons under trial during the year', 'persons against whom cases were compounded or withdrawn', 'persons in custody or on bail during the stage of trial at the end of the year', 'persons in whose cases trials were completed during the year', 'persons convicted', 'persons acquitted', 'unnamed: 14', 'unnamed: 15', 'unnamed: 16', 'unnamed: 17', 'unnamed: 18', 'unnamed: 19', 'unnamed: 20', 'unnamed: 21', 'unnamed: 22', 'unnamed: 23', 'unnamed: 24', 'unnamed: 25', 'unnamed: 26', 'unnamed: 27', 'unnamed: 28', 'unnamed: 29', 'unnam

In [53]:
arrests["year"] = 2014   # change to correct year if needed

In [54]:
arrests = arrests.rename(columns={
    "persons arrested during the year": "total_arrests"
})

In [55]:
arrests = arrests[["state","year","total_arrests"]]
arrests = arrests.groupby(["state","year"])["total_arrests"].sum().reset_index()


In [56]:
df = ipc
df = df.merge(women, on=["state","district","year"], how="left")
df = df.merge(children, on=["state","district","year"], how="left")
df = df.merge(police, on=["state","year"], how="left")
df = df.merge(arrests, on=["state","year"], how="left")




In [57]:
print(df[["total_police","total_arrests"]].head(20))

    total_police  total_arrests
0       326924.0            NaN
1       326924.0            NaN
2       326924.0            NaN
3       326924.0            NaN
4       326924.0            NaN
5       326924.0            NaN
6       326924.0            NaN
7       326924.0            NaN
8       326924.0            NaN
9       326924.0            NaN
10      326924.0            NaN
11      326924.0            NaN
12      326924.0            NaN
13      326924.0            NaN
14      326924.0            NaN
15      326924.0            NaN
16      326924.0            NaN
17      326924.0            NaN
18      326924.0            NaN
19      326924.0            NaN


In [58]:
df = df.fillna(0)
df = df[df["year"] >= 2012]

In [70]:
df = df[[
    "state","district","year",
    "ipc_total","women_total","children_total",
    "total_police","total_arrests"
]]

In [71]:
df["total_crime"] = df["ipc_total"] + df["women_total"] + df["children_total"]
df["crime_per_police"] = df["total_crime"] / (df["total_police"] + 1)
df["arrest_rate"] = df["total_arrests"] / (df["total_crime"] + 1)

In [72]:
def risk(x):
    if x > 40000:
        return "High"
    elif x > 20000:
        return "Medium"
    else:
        return "Low"

df["risk"] = df["total_crime"].apply(risk)

In [73]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[[
    "total_police",
    "total_arrests"
]]
y = df["risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = RandomForestClassifier(max_depth=5)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.8626262626262626


In [90]:
def predict_crime_count(state):
    
    state = state.strip().upper()
    
    if state not in df["state"].unique():
        return "State not found", None, None
    
    state_data = df[df["state"] == state]
    
    police = state_data["total_police"].mean()
    arrests = state_data["total_arrests"].mean()
    
    # ✅ FIX HERE
    input_data = pd.DataFrame({
        "total_police": [police],
        "total_arrests": [arrests]
    })
    
    predicted_crime = model.predict(input_data)[0]
    
    return predicted_crime, police, arrests

In [91]:
def predict_state_full(state):
    
    state = state.strip().upper()
    
    if state not in df["state"].unique():
        return "State not found", None, None, None
    
    state_data = df[df["state"] == state]
    
    # Aggregate
    police = state_data["total_police"].mean()
    arrests = state_data["total_arrests"].mean()
    
    # Create input (IMPORTANT - fixes warning)
    input_data = pd.DataFrame({
        "total_police": [police],
        "total_arrests": [arrests]
    })
    
    # Predictions
    risk = model.predict(input_data)[0]
    crime = model.predict(input_data)[0]
    
    return risk, int(crime), police, arrests

In [92]:
def give_reason(police, arrests):
    
    if police < 3000:
        return "Low police strength may lead to higher crime"
    
    elif arrests < 1000:
        return "Low arrest rate indicates weak law enforcement"
    
    else:
        return "Balanced policing and arrest activity"

In [93]:
def give_suggestions(risk):
    if risk == "High":
        return [
            "Increase police deployment",
            "Install CCTV cameras",
            "Improve surveillance"
        ]
    elif risk == "Medium":
        return [
            "Increase patrol",
            "Monitor sensitive zones"
        ]
    else:
        return [
            "Maintain safety measures"
        ]

In [94]:
state = input("Enter State: ")

result = predict_state_full(state)

if result[0] == "State not found":
    print("❌ State not found in dataset")

else:
    risk, crime, police, arrests = result
    
    print("\nPredicted Risk:", risk)
    print("Predicted Crime Count:", crime)
    
    reason = give_reason(police, arrests)
    print("Reason:", reason)
    
    print("\nSuggestions:")
    for s in give_suggestions(risk):
        print("-", s)

Enter State:  Andhra pradesh


ValueError: invalid literal for int() with base 10: 'Low'

In [67]:
print("Available States:")
print(df["state"].unique())

print("\nAvailable Districts:")
print(df["district"].unique())

Available States:
['ANDHRA PRADESH' 'ARUNACHAL PRADESH' 'ASSAM' 'BIHAR' 'CHHATTISGARH' 'GOA'
 'GUJARAT' 'HARYANA' 'HIMACHAL PRADESH' 'JAMMU AND KASHMIR' 'JHARKHAND'
 'KARNATAKA' 'KERALA' 'MADHYA PRADESH' 'MAHARASHTRA' 'MANIPUR' 'MEGHALAYA'
 'MIZORAM' 'NAGALAND' 'ODISHA' 'PUNJAB' 'RAJASTHAN' 'SIKKIM' 'TAMIL NADU'
 'TRIPURA' 'UTTAR PRADESH' 'UTTARAKHAND' 'WEST BENGAL' 'A AND N ISLANDS'
 'CHANDIGARH' 'D AND N HAVELI' 'DAMAN AND DIU' 'DELHI UT' 'LAKSHADWEEP'
 'PUDUCHERRY' 'AANDN ISLANDS' 'DANDN HAVELI' 'TELANGANA']

Available Districts:
['ADILABAD' 'ANANTAPUR' 'CHITTOOR' ... 'Lakshadweep' 'Karaikal'
 'Puducherry']
